# CS781 Phase 1 — Model 1 (GQA + RoPE) — Training

**Inputs to attach** (Add Data): (1) the tokenizer dataset, (2) the output of the tokenization notebook (`shards/`, `val_tokens.npy`, `tokenized_meta.json`), and, from the second session on, (3) the previous version of *this* notebook's output.

**Per run you only change `STEPS_THIS_RUN` (and `TOKEN_BUDGET` while calibrating).** The LR schedule is fixed to the whole training length, each session just stops early and checkpoints, and the next session resumes automatically.

Needs no internet and no Hugging Face login.

## 1. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
N_GPU = max(1, torch.cuda.device_count())

## 2. Configuration

In [ ]:
import os, json, glob, time, math, shutil
import numpy as np

SESSION_START = time.time()

# ===== change these per run =====
STEPS_THIS_RUN = 2      # optimizer steps for THIS session. 2 = quick timing/diagnostic check. None = run until the
                        # time limit below (or the end of training), whichever comes first
TOKEN_BUDGET = 130_000_000  # training tokens to use. Calibration: small. Real run: set from the printed suggestion
MAX_SESSION_HOURS = 11.0  # training stops + checkpoints after this long, so one commit always ends
                          # cleanly under the 12h limit (leaves time for final eval + save)

# ===== fixed for the whole training: do NOT change between sessions =====
SEED = 42
BLOCK = 1024                      # training sequence length (tokens)
N_EMBD, N_LAYER, N_HEAD, N_KV_HEAD, D_FF = 1024, 8, 16, 4, 2688
PER_DEVICE_BATCH = 8
GRAD_ACCUM_STEPS = 32             # 8 seqs x 2 GPUs x 32 = 512 seqs = 524,288 tokens per optimizer step
LEARNING_RATE = 3e-4
WARMUP_STEPS = 300
SAVE_STEPS = 100                  # checkpoint every N optimizer steps
EVAL_STEPS = 200

# ---- data mix (fixed for the whole training) ----
SOURCES = ["acts", "high_courts", "supreme_court", "tribunals"]   # the order the tokenization notebook read the files in
SOURCE_DOCS = {"acts": 18_826, "high_courts": 2_536_738, "supreme_court": 55_549, "tribunals": 869_132}
MIX = "equal"                     # "equal" | "proportional" | {"acts": 1, "high_courts": 2, ...} (relative weights)
VAL_RUNS, VAL_RUN_BLOCKS, VAL_MARGIN_BLOCKS = 25, 10, 8   # per source: 25 runs x 10 blocks held out, margin kept out of training
TOTAL_GPU_HOUR_BUDGET = 30.0
TRAIN_BUDGET_HOURS = 28.0         # plan the whole run to fit this, leaving slack under the 30h

OUTPUT_DIR = "/kaggle/working/cs781_phase1_model1"
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "legal-gqa-final")
TRAINING_META_PATH = os.path.join(OUTPUT_DIR, "training_meta.json")
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

## 3. Load Tokenizer and Token Shards

Inputs are located by searching `/kaggle/input`, so the exact mount path doesn't matter. The training data is memory-mapped shard by shard and sliced into blocks on demand: nothing is concatenated or copied into RAM.

In [ ]:
from transformers import AutoTokenizer

def show_inputs():
    print("Attached inputs under /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root[len("/kaggle/input"):].count(os.sep)
        print("  " * depth + (os.path.basename(root) or "input") + "/", f"({len(files)} files)" if files else "")
        if depth >= 3:
            dirs[:] = []

def find_dirs(filename):
    return sorted({os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{filename}", recursive=True)})

# tokenizer.json also appears inside this notebook's own saved checkpoints; ignore those
cands = [d for d in find_dirs("tokenizer.json") if "checkpoint-" not in d and "legal-gqa-final" not in d]
if not cands:
    show_inputs()
    raise FileNotFoundError("No tokenizer found. Attach your tokenizer DATASET to this notebook (Add Input); "
                            "the listing above shows what is currently attached.")
TOKENIZER_DIR = ([d for d in cands if os.path.exists(os.path.join(d, "trained_on.json"))] or cands)[0]
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)
print("Tokenizer:", TOKENIZER_DIR, "| vocab size:", len(tokenizer))

meta_dirs = find_dirs("tokenized_meta.json")
if not meta_dirs:
    show_inputs()
    raise FileNotFoundError("No tokenized_meta.json found. Attach the OUTPUT of your tokenization notebook to this notebook.")
TOKEN_ROOT = meta_dirs[0]
META_PATH = os.path.join(TOKEN_ROOT, "tokenized_meta.json")
meta = json.load(open(META_PATH))
assert meta["vocab_size"] == len(tokenizer), "token shards were built with a different tokenizer!"
print(f"Token data: {TOKEN_ROOT} | {meta['train_tokens']:,} train tokens in {len(meta['shards'])} shards")

## 3b. Build the Training Mix and Validation Set

The shards are in file order (acts, then high courts, then supreme court, then tribunals), so taking a slice of them would train on one source only. This cell recovers where each source starts (by counting end-of-document tokens, no re-tokenizing) and then draws a random subset from across each *whole* source. `MIX = "equal"` gives every source the same number of tokens where the data allows it; a small source is used in full and the leftover goes to the others. Validation is held out per source, as random contiguous runs with a margin kept out of training so the same document isn't in both.

In [ ]:
import torch

# ---- 1. open every shard (memory-mapped: nothing is loaded into RAM) ----
arrays = [np.load(os.path.join(TOKEN_ROOT, "shards", s), mmap_mode="r") for s in meta["shards"]]
eos = tokenizer.eos_token_id
shard_len = np.array([len(a) for a in arrays])
tok_off = np.concatenate([[0], np.cumsum(shard_len)])          # global token offset where each shard starts
n_blk = shard_len // BLOCK
cum_blk = np.concatenate([[0], np.cumsum(n_blk)])              # global id of each shard's first block
N_BLOCKS = int(cum_blk[-1])

# ---- 2. where does each source start? (count end-of-document tokens) ----
t_scan = time.time()
assert sum(SOURCE_DOCS.values()) == meta["num_docs"], "SOURCE_DOCS doesn't add up to the tokenized corpus"
val_arr = np.load(os.path.join(TOKEN_ROOT, "val_tokens.npy"))
n_val_docs = int((val_arr == eos).sum()) + (0 if int(val_arr[-1]) == eos else 1)   # tokenization sent every 20th doc to val until full
eos_per_shard = np.array([int(np.count_nonzero(a == eos)) for a in arrays])
cum_eos = np.concatenate([[0], np.cumsum(eos_per_shard)])
assert int(cum_eos[-1]) == meta["num_docs"] - n_val_docs, (
    f"doc count mismatch: {int(cum_eos[-1]):,} end-of-doc tokens in train shards vs "
    f"{meta['num_docs']:,} docs - {n_val_docs:,} val docs. The token data isn't what this cell assumes.")

def train_doc_index(g):
    # global doc index -> index among train docs (val docs were global indices 0, 20, 40, ... n_val_docs of them)
    return g - min(n_val_docs, -(-g // 20))

def token_offset_of_train_doc(t):
    # global token offset where the t-th train doc starts = just after the t-th end-of-document token
    if t == 0:
        return 0
    j = int(np.searchsorted(cum_eos, t, side="left")) - 1
    pos = np.flatnonzero(arrays[j] == eos)
    return int(tok_off[j] + pos[t - int(cum_eos[j]) - 1] + 1)

doc_edges = np.cumsum([0] + [SOURCE_DOCS[s] for s in SOURCES])
src_start_tok = np.array([0] + [token_offset_of_train_doc(train_doc_index(int(g))) for g in doc_edges[1:-1]])

# ---- 3. which source does every block belong to? ----
shard_of_blk = np.repeat(np.arange(len(arrays)), n_blk)
blk_start_tok = tok_off[shard_of_blk] + (np.arange(N_BLOCKS) - cum_blk[shard_of_blk]) * BLOCK
blk_source = np.searchsorted(src_start_tok[1:], blk_start_tok, side="right")     # 0..len(SOURCES)-1, non-decreasing
src_lo = np.searchsorted(blk_source, np.arange(len(SOURCES)), side="left")
src_hi = np.searchsorted(blk_source, np.arange(len(SOURCES)), side="right")

# ---- 4. hold out validation: evenly spread random runs per source, with a margin excluded from training ----
rng = np.random.default_rng(SEED)
excluded = np.zeros(N_BLOCKS, dtype=bool)
val_ids = {}
for si, s in enumerate(SOURCES):
    lo, hi = int(src_lo[si]), int(src_hi[si])
    stratum = (hi - lo) // VAL_RUNS
    assert stratum > VAL_RUN_BLOCKS + 2 * VAL_MARGIN_BLOCKS + 1, f"source {s} is too small for {VAL_RUNS} validation runs"
    runs = []
    for r in range(VAL_RUNS):
        a = lo + r * stratum + VAL_MARGIN_BLOCKS + int(rng.integers(0, stratum - VAL_RUN_BLOCKS - 2 * VAL_MARGIN_BLOCKS))
        runs.append(np.arange(a, a + VAL_RUN_BLOCKS))
        excluded[a - VAL_MARGIN_BLOCKS: a + VAL_RUN_BLOCKS + VAL_MARGIN_BLOCKS] = True
    val_ids[s] = np.concatenate(runs)

# ---- 5. how many blocks does each source contribute? ----
cand = {s: np.arange(src_lo[i], src_hi[i])[~excluded[src_lo[i]:src_hi[i]]] for i, s in enumerate(SOURCES)}
avail = {s: len(cand[s]) for s in SOURCES}

def allocate(avail, weights, total):
    # split `total` blocks across sources in proportion to `weights`, never exceeding what a source has;
    # whatever a small source can't absorb is redistributed over the others
    alloc = {s: 0 for s in avail}
    active = [s for s in avail if weights[s] > 0]
    remaining = int(total)
    while active and remaining > 0:
        wsum = sum(weights[s] for s in active)
        share = {s: remaining * weights[s] / wsum for s in active}
        capped = [s for s in active if share[s] >= avail[s]]
        if not capped:
            for s in active:
                alloc[s] = int(share[s])
            break
        for s in capped:
            alloc[s] = avail[s]
            remaining -= avail[s]
            active.remove(s)
    return alloc

if MIX == "equal":
    weights = {s: 1.0 for s in SOURCES}
elif MIX == "proportional":
    weights = {s: float(avail[s]) for s in SOURCES}
else:
    weights = {s: float(MIX[s]) for s in SOURCES}
alloc = allocate(avail, weights, TOKEN_BUDGET // BLOCK)
train_ids = rng.permutation(np.concatenate([rng.choice(cand[s], size=alloc[s], replace=False) for s in SOURCES]))

# ---- 6. datasets ----
class PickedBlocks(torch.utils.data.Dataset):
    """Fixed-length blocks, read on demand from the memory-mapped shards by global block id."""
    def __init__(self, ids):
        self.ids = np.asarray(ids)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        g = int(self.ids[i])
        j = int(np.searchsorted(cum_blk, g, side="right") - 1)
        k = g - int(cum_blk[j])
        return {"input_ids": arrays[j][k * BLOCK:(k + 1) * BLOCK].astype(np.int64).tolist()}

train_dataset = PickedBlocks(train_ids)
val_by_source = {s: PickedBlocks(val_ids[s]) for s in SOURCES}
val_dataset = PickedBlocks(np.concatenate([val_ids[s] for s in SOURCES]))

GLOBAL_SEQS = PER_DEVICE_BATCH * N_GPU * GRAD_ACCUM_STEPS
TOTAL_TRAIN_STEPS = len(train_dataset) // GLOBAL_SEQS          # one pass over the data

print(f"scanned {len(arrays)} shards in {time.time() - t_scan:.0f}s | {meta['num_docs']:,} docs, {n_val_docs:,} of them were held out by the tokenizer")
print(f"MIX = {MIX!r} | budget {TOKEN_BUDGET/1e6:,.0f}M tokens (corpus has {N_BLOCKS * BLOCK / 1e9:.2f}B)")
print(f"{'source':14s}{'available':>11s}{'used':>10s}{'share':>8s}{'of source':>11s}")
_tot = max(sum(alloc.values()), 1)
for s in SOURCES:
    print(f"{s:14s}{avail[s] * BLOCK / 1e6:10.0f}M{alloc[s] * BLOCK / 1e6:9.0f}M{100 * alloc[s] / _tot:7.1f}%{100 * alloc[s] / max(avail[s], 1):10.1f}%")
print(f"validation: {len(val_dataset):,} blocks ({VAL_RUNS * VAL_RUN_BLOCKS} per source, in {VAL_RUNS} runs of {VAL_RUN_BLOCKS})")
print(f"Train blocks: {len(train_dataset):,} | Tokens/step: {GLOBAL_SEQS * BLOCK:,} | TOTAL_TRAIN_STEPS (one epoch): {TOTAL_TRAIN_STEPS:,}")
if TOKEN_BUDGET < 1_000_000_000:
    print("NOTE: small TOKEN_BUDGET -> calibration only. Set it from the printed suggestion for the real run.")

## 4. Define The Model

Written to a real file and imported: recent `transformers` reads `__file__` from a model class's module, and a class defined inside a notebook cell has none.

In [ ]:
%%writefile /kaggle/working/legal_gqa_model.py
"""
Model 1 for CS781 Phase-1: GQA + RoPE + SwiGLU + RMSNorm decoder-only transformer.

Wrapped as a transformers PretrainedConfig/PreTrainedModel subclass so it is a
drop-in replacement for GPT2Config/GPT2LMHeadModel in the starter notebook:
same save_pretrained/from_pretrained, same Trainer + TrainingArguments +
resume_from_checkpoint behavior, same model.generate() call signature.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast


class LegalGQAConfig(PretrainedConfig):
    model_type = "legal_gqa"

    def __init__(
        self,
        vocab_size=50257,
        n_positions=1024,
        n_embd=1024,
        n_layer=8,
        n_head=16,
        n_kv_head=4,
        d_ff=2688,
        rope_theta=10000.0,
        dropout=0.1,
        initializer_range=0.02,
        rms_norm_eps=1e-5,
        bos_token_id=50256,
        eos_token_id=50256,
        pad_token_id=50256,
        activation_checkpointing=False,
        tie_word_embeddings=True,
        **kwargs,
    ):
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.vocab_size = vocab_size
        self.n_positions = n_positions
        self.n_embd = n_embd
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.d_ff = d_ff
        self.rope_theta = rope_theta
        self.dropout = dropout
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.use_cache = True
        # own implementation (HF's gradient_checkpointing flag is not supported here)
        self.activation_checkpointing = activation_checkpointing
        super().__init__(
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            tie_word_embeddings=tie_word_embeddings,
            **kwargs,
        )


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k, cos, sin):
    # q, k: (B, n_head, T, head_dim); cos/sin: (T, head_dim)
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    q_rot = (q * cos) + (rotate_half(q) * sin)
    k_rot = (k * cos) + (rotate_half(k) * sin)
    return q_rot, k_rot


class GroupedQueryAttention(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.head_dim = config.n_embd // config.n_head
        self.n_rep = self.n_head // self.n_kv_head
        self.dropout_p = config.dropout

        self.q_proj = nn.Linear(config.n_embd, config.n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.n_head * self.head_dim, config.n_embd, bias=False)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_key_value=None, use_cache=False):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        q, k = apply_rope(q, k, cos, sin)
        # RoPE tables are fp32, v comes out of a linear (fp16 under autocast): force one dtype
        q, k = q.to(v.dtype), k.to(v.dtype)

        if past_key_value is not None:
            past_k, past_v = past_key_value
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        present = (k, v) if use_cache else None

        # GQA: broadcast each KV head across its group of query heads
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_bias,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=is_causal,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim)
        return self.o_proj(out), present


class SwiGLU(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.gate_proj = nn.Linear(config.n_embd, config.d_ff, bias=False)
        self.up_proj = nn.Linear(config.n_embd, config.d_ff, bias=False)
        self.down_proj = nn.Linear(config.d_ff, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class DecoderBlock(nn.Module):
    def __init__(self, config: LegalGQAConfig):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.attn = GroupedQueryAttention(config)
        self.ln_2 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.ffn = SwiGLU(config)

    def forward(self, x, cos, sin, attn_bias, is_causal, past_key_value=None, use_cache=False):
        attn_out, present = self.attn(
            self.ln_1(x), cos, sin, attn_bias, is_causal, past_key_value, use_cache
        )
        x = x + attn_out
        x = x + self.ffn(self.ln_2(x))
        return x, present


class LegalGQAForCausalLM(PreTrainedModel):
    config_class = LegalGQAConfig
    # input embedding and output head share one weight; must be declared or
    # save_pretrained / Trainer checkpointing refuses to save
    _tied_weights_keys = {"lm_head.weight": "wte.weight"}

    def __init__(self, config: LegalGQAConfig):
        super().__init__(config)
        self.head_dim = config.n_embd // config.n_head

        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # weight tying

        self.post_init()

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        # scaled init on residual-stream output projections (GPT-2 recipe)
        for name, p in module.named_parameters(recurse=False):
            if name in ("o_proj.weight", "down_proj.weight"):
                nn.init.normal_(
                    p, mean=0.0, std=std / math.sqrt(2 * self.config.n_layer)
                )

    def get_input_embeddings(self):
        return self.wte

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.lm_head = new_embeddings

    def set_input_embeddings(self, value):
        self.wte = value

    def _rope_slice(self, past_len, T, device, dtype):
        # Computed on the fly (float32, then cast). Deliberately NOT stored as a buffer:
        # non-persistent buffers come back uninitialized after from_pretrained().
        inv_freq = 1.0 / (
            self.config.rope_theta
            ** (torch.arange(0, self.head_dim, 2, device=device).float() / self.head_dim)
        )
        pos = torch.arange(past_len, past_len + T, device=device).float()
        emb = torch.cat((torch.outer(pos, inv_freq),) * 2, dim=-1)
        return emb.cos().to(dtype), emb.sin().to(dtype)

    def _build_attn_bias(self, attention_mask, past_len, T, device, dtype):
        """Returns (attn_bias, is_causal). attn_bias is None when the fast
        SDPA is_causal fused-kernel path is usable (no padding to mask)."""
        no_padding = attention_mask is None or bool(attention_mask.bool().all())
        if no_padding:
            return None, T > 1

        total_len = past_len + T
        pad = attention_mask[:, None, None, :].to(dtype)
        bias = (1.0 - pad) * torch.finfo(dtype).min

        if T > 1:
            q_pos = torch.arange(past_len, past_len + T, device=device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=device).unsqueeze(0)
            causal = (k_pos > q_pos).to(dtype) * torch.finfo(dtype).min
            bias = bias + causal[None, None, :, :]
        return bias, False

    def forward(
        self,
        input_ids,
        attention_mask=None,
        past_key_values=None,
        labels=None,
        use_cache=None,
    ):
        # NOTE: no **kwargs here on purpose. If forward() accepts **kwargs, Trainer assumes the
        # model normalises its own loss, then multiplies it by n_gpu and skips the
        # gradient-accumulation division (loss reported 2x-64x too high).
        #
        # fp16 autocast is switched on HERE rather than relying on Trainer: under nn.DataParallel
        # on Kaggle the matmuls were measurably running in fp32 (no tensor cores, ~4x slower).
        # Only used for unpadded batches (all of training/eval); padded generation keeps the
        # exact fp32 path, since fp16 additive masks can overflow to -inf.
        amp = input_ids.is_cuda and (attention_mask is None or bool(attention_mask.bool().all()))
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            return self._forward_impl(input_ids, attention_mask, past_key_values, labels, use_cache)

    def _forward_impl(self, input_ids, attention_mask, past_key_values, labels, use_cache):
        use_cache = self.config.use_cache if use_cache is None else use_cache
        if self.training:
            use_cache = False  # never keep K/V during training
        B, T = input_ids.shape
        past_len = past_key_values[0][0].shape[2] if past_key_values is not None else 0

        x = self.drop(self.wte(input_ids))
        cos, sin = self._rope_slice(past_len, T, input_ids.device, x.dtype)
        attn_bias, is_causal = self._build_attn_bias(
            attention_mask, past_len, T, input_ids.device, x.dtype
        )

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            pkv = past_key_values[i] if past_key_values is not None else None
            if self.training and self.config.activation_checkpointing:
                x = torch_checkpoint(
                    lambda h, blk=block: blk(h, cos, sin, attn_bias, is_causal, None, False)[0],
                    x,
                    use_reentrant=False,
                )
            else:
                x, present = block(x, cos, sin, attn_bias, is_causal, pkv, use_cache)
                if use_cache:
                    presents.append(present)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        # when labels are given only the loss is needed; returning the full
        # (B, T, vocab) logits would just be gathered across GPUs for nothing
        return CausalLMOutputWithPast(
            loss=loss,
            logits=None if labels is not None else logits,
            past_key_values=tuple(presents) if use_cache else None,
        )

    # No prepare_inputs_for_generation / GenerationMixin here on purpose —
    # generation is done by generate_utils.generate_batch(), which calls
    # this model's forward() directly with its own tuple-based KV cache
    # rather than going through model.generate().

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")
from legal_gqa_model import LegalGQAConfig, LegalGQAForCausalLM

config = LegalGQAConfig(
    vocab_size=len(tokenizer), n_positions=BLOCK,
    n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD, d_ff=D_FF,
    bos_token_id=tokenizer.bos_token_id, eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    activation_checkpointing=True,      # trades ~30% speed for a lot of activation memory
)
model = LegalGQAForCausalLM(config)
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

## 5. Training Arguments

`max_steps` is the **whole** training length, so the cosine schedule is one continuous curve across sessions. `StopAfter` ends *this session* early and forces a checkpoint.

In [ ]:
from transformers import (TrainingArguments, Trainer, TrainerCallback,
                          DataCollatorForLanguageModeling)
from transformers.trainer_utils import get_last_checkpoint

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    max_steps=TOTAL_TRAIN_STEPS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH * 2,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.1,
    adam_beta1=0.9, adam_beta2=0.95,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr_rate": 0.1},
    fp16=torch.cuda.is_available(),      # T4 is Turing: fp16, not bf16
    max_grad_norm=1.0,
    eval_strategy="steps", eval_steps=EVAL_STEPS, prediction_loss_only=True,
    logging_steps=10,
    save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=2,
    dataloader_num_workers=2,
    seed=SEED,
    report_to="none",
    disable_tqdm=True,   # the notebook progress widget has a bug that crashes a session ending at exactly 2 steps; plain-text logs suit a commit anyway
)

class StopAfter(TrainerCallback):
    """Ends THIS session (with a checkpoint) at a step count OR a wall-clock deadline."""
    def __init__(self, stop_step, deadline):
        self.stop_step, self.deadline = stop_step, deadline
    def on_step_end(self, args, state, control, **kwargs):
        out_of_time = time.time() > self.deadline
        if state.global_step >= self.stop_step or out_of_time:
            if out_of_time and state.global_step < self.stop_step:
                print(f"\nSession time limit reached at step {state.global_step}: checkpointing and stopping.")
            control.should_save = True
            control.should_training_stop = True
        return control

## 6. Train (auto-resume + per-step timing)

In [ ]:
# --- pull the newest checkpoint from a previously attached output, if we have none yet ---
if not glob.glob(os.path.join(MODEL_OUTPUT_DIR, "checkpoint-*")):
    prev = glob.glob("/kaggle/input/**/legal-gqa-final/checkpoint-*", recursive=True)
    if prev:
        latest = max(prev, key=lambda p: int(p.rsplit("-", 1)[1]))
        shutil.copytree(latest, os.path.join(MODEL_OUTPUT_DIR, os.path.basename(latest)))
        prev_meta = os.path.join(os.path.dirname(os.path.dirname(latest)), "training_meta.json")
        if os.path.exists(prev_meta):
            shutil.copy(prev_meta, TRAINING_META_PATH)
        print("Restored checkpoint from previous output:", latest)

# --- refuse to resume into a differently-configured run ---
fingerprint = {"global_seqs": GLOBAL_SEQS, "block": BLOCK, "total_steps": TOTAL_TRAIN_STEPS,
               "arch": [N_EMBD, N_LAYER, N_HEAD, N_KV_HEAD, D_FF], "lr": LEARNING_RATE, "warmup": WARMUP_STEPS,
               "data": [TOKEN_BUDGET, str(MIX), SEED, VAL_RUNS, VAL_RUN_BLOCKS, VAL_MARGIN_BLOCKS]}
cumulative_seconds = 0.0
if os.path.exists(TRAINING_META_PATH):
    saved = json.load(open(TRAINING_META_PATH))
    cumulative_seconds = saved.get("cumulative_seconds", 0.0)
    if saved.get("fingerprint") != fingerprint and glob.glob(os.path.join(MODEL_OUTPUT_DIR, "checkpoint-*")):
        raise RuntimeError(f"Checkpoint was made with a different config!\n saved:   {saved.get('fingerprint')}\n current: {fingerprint}\n"
                           "Either restore the old settings or start fresh (don't attach the old output).")

last_checkpoint = get_last_checkpoint(MODEL_OUTPUT_DIR)
steps_at_start = int(last_checkpoint.rsplit("-", 1)[1]) if last_checkpoint else 0
stop_step = TOTAL_TRAIN_STEPS if STEPS_THIS_RUN is None else min(steps_at_start + STEPS_THIS_RUN, TOTAL_TRAIN_STEPS)
deadline = SESSION_START + MAX_SESSION_HOURS * 3600
print(f"Resuming from step {steps_at_start}" if last_checkpoint else "Starting fresh.")
print(f"This session runs to step {stop_step} of {TOTAL_TRAIN_STEPS} (or until {MAX_SESSION_HOURS}h of session time)")

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    data_collator=data_collator, processing_class=tokenizer,
    callbacks=[StopAfter(stop_step, deadline)],
)

# ---- diagnostics: is the GPU busy, and are matmuls really running in fp16? ----
import subprocess, threading
gpu_samples, probe = [], {}
def _num(v):
    try: return float(v)
    except ValueError: return float("nan")
def _monitor(stop):
    while not stop.is_set():
        try:
            out = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,power.draw,clocks.sm",
                 "--format=csv,noheader,nounits"], text=True, timeout=10)
            gpu_samples.append([[_num(v.strip()) for v in ln.split(",")] for ln in out.strip().splitlines()])
        except Exception:
            return
        stop.wait(5)
_stop = threading.Event()
threading.Thread(target=_monitor, args=(_stop,), daemon=True).start()
def _record_dtype(module, inputs, output):
    probe.setdefault("q_proj output dtype", str(output.dtype))   # returns None: must not replace the output
_hook = model.blocks[0].attn.q_proj.register_forward_hook(_record_dtype)

t0 = time.time()
trainer.train(resume_from_checkpoint=last_checkpoint)
elapsed = time.time() - t0

_stop.set(); _hook.remove()
print("\n--- diagnostics ---")
print("matmul output dtype inside the model:", probe.get("q_proj output dtype", "n/a"), "(float16 = tensor cores in use)")
if gpu_samples:
    for g in range(len(gpu_samples[0])):
        col = lambda k: [smp[g][k] for smp in gpu_samples if len(smp) > g and smp[g][k] == smp[g][k]]
        m = lambda xs: sum(xs) / len(xs) if xs else float("nan")
        print(f"GPU {g}: util {m(col(1)):.0f}% | mem peak {max(col(2) or [0]):.0f} MB | power {m(col(3)):.0f} W | SM clock {m(col(4)):.0f} MHz  ({len(gpu_samples)} samples)")
else:
    print("nvidia-smi sampling unavailable")
if torch.cuda.is_available():
    print("torch peak allocated:", [f"{torch.cuda.max_memory_allocated(i)/2**30:.1f} GB" for i in range(torch.cuda.device_count())])

steps_done = trainer.state.global_step - steps_at_start
sec_per_step = elapsed / max(steps_done, 1)
cumulative_seconds += elapsed
json.dump({"cumulative_seconds": cumulative_seconds, "global_step": trainer.state.global_step,
           "fingerprint": fingerprint}, open(TRAINING_META_PATH, "w"))

remaining_steps = TOTAL_TRAIN_STEPS - trainer.state.global_step
print()
print(f"Steps this session: {steps_done} | wall time: {elapsed/60:.1f} min | {sec_per_step:.1f} s/step")
print(f"Progress: step {trainer.state.global_step:,} / {TOTAL_TRAIN_STEPS:,}")
print(f"GPU time used so far: {cumulative_seconds/3600:.2f} / {TOTAL_GPU_HOUR_BUDGET:.0f} h")
print(f"Full remaining training at this speed: {remaining_steps * sec_per_step / 3600:.1f} h")

# ---- planning helper: how much data can the whole run afford at this speed? ----
budget_steps = int(TRAIN_BUDGET_HOURS * 3600 / sec_per_step)
fit_tokens = budget_steps * GLOBAL_SEQS * BLOCK
print()
print(f"At {sec_per_step:.1f} s/step, {TRAIN_BUDGET_HOURS:.0f}h of training = {budget_steps:,} steps = {budget_steps * GLOBAL_SEQS * BLOCK / 1e9:.2f}B tokens")
print(f"  -> real run: set TOKEN_BUDGET = {int(fit_tokens // 10_000_000 * 10_000_000):,}  (the corpus has {N_BLOCKS * BLOCK / 1e9:.2f}B)")
print(f"  -> ~{math.ceil(TRAIN_BUDGET_HOURS / MAX_SESSION_HOURS)} sessions of up to {MAX_SESSION_HOURS:.0f}h; set STEPS_THIS_RUN = None and each one stops itself in time")
if steps_done <= 3:
    print("(few steps measured: first-step overhead inflates this; treat as an upper bound)")

## 7. Validation Loss, Loss Plot, Save Model

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

eval_loss = trainer.evaluate()["eval_loss"]
print(f"Validation loss: {eval_loss:.4f} | perplexity: {math.exp(eval_loss):.2f}")

logs = trainer.state.log_history
tr = [(l["step"], l["loss"]) for l in logs if "loss" in l]
ev = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]
plt.figure(figsize=(9, 4.5))
if tr: plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), label="val loss")
plt.xlabel("optimizer step"); plt.ylabel("loss"); plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss.png"), dpi=150)

print("Per-source validation:")
for s in SOURCES:
    r = trainer.evaluate(eval_dataset=val_by_source[s], metric_key_prefix=f"val_{s}")
    print(f"  {s:14s} loss {r[f'val_{s}_loss']:.4f} | perplexity {math.exp(r[f'val_{s}_loss']):.1f}")

trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print("Saved model + tokenizer to", MODEL_OUTPUT_DIR)